In [ ]:
# Подключает ваш Google Drive к виртуальной машине Colab.
from google.colab import drive
drive.mount('/content/drive')

# TODO: Укажите имя папки в вашем Drive, в которой вы сохранили распакованный
# каталог задания, например 'cs231n/assignments/assignment2/'
FOLDERNAME = 'cs231n/assignments/assignment2/'
assert FOLDERNAME is not None, "[!] Введите имя папки."

# Теперь, когда Drive подключён, это даёт возможность
# интерпретатору Python виртуальной машины Colab загружать
# python-файлы изнутри него.
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# Эта команда скачивает набор данных CIFAR-10 на ваш Drive
# если он ещё не существует.
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

# Сверточные сети

До сих пор мы работали с глубоко связанными полносвязными сетями, используя их для изучения различных стратегий оптимизации и архитектур сетей. Полносвязные сети являются хорошей площадкой для экспериментов, потому что они очень вычислительно эффективны, но на практике все современные результаты достигаются с использованием сверточных сетей.

Сначала вы реализуете несколько типов слоёв, которые используются в сверточных сетях. Затем вы будете использовать эти слои для обучения сверточной сети на наборе данных CIFAR-10.

In [ ]:
# Ячейка настройки.
import numpy as np
import matplotlib.pyplot as plt
from cs231n.classifiers.cnn import *
from cs231n.data_utils import get_CIFAR10_data
from cs231n.gradient_check import eval_numerical_gradient_array, eval_numerical_gradient
from cs231n.layers import *
from cs231n.fast_layers import *
from cs231n.solver import Solver

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # задаём размер графиков по умолчанию
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

import sys
import types
import importlib

if "imp" not in sys.modules:
    imp = types.ModuleType("imp")
    imp.reload = importlib.reload
    sys.modules["imp"] = imp

%load_ext autoreload
%autoreload 2

def rel_error(x, y):
  """ возвращает относительную ошибку """
  return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

In [ ]:
# Загружаем предобработанные данные CIFAR-10.
data = get_CIFAR10_data()
for k, v in list(data.items()):
    print(f"{k}: {v.shape}")

# Свертка: прямой проход без оптимизаций
Основой сверточной сети является операция свёртки. В файле `cs231n/layers.py` реализуйте прямой проход для слоя свёртки в функции `conv_forward_naive`.

На этом этапе вам не нужно слишком беспокоиться об эффективности; просто напишите код так, как вам кажется наиболее понятным.

Проверить реализацию можно, запустив следующее:

In [ ]:
x_shape = (2, 3, 4, 4)
w_shape = (3, 3, 4, 4)
x = np.linspace(-0.1, 0.5, num=np.prod(x_shape)).reshape(x_shape)
w = np.linspace(-0.2, 0.3, num=np.prod(w_shape)).reshape(w_shape)
b = np.linspace(-0.1, 0.2, num=3)

conv_param = {'stride': 2, 'pad': 1}
out, _ = conv_forward_naive(x, w, b, conv_param)
correct_out = np.array([[[[-0.08759809, -0.10987781],
                           [-0.18387192, -0.2109216 ]],
                          [[ 0.21027089,  0.21661097],
                           [ 0.22847626,  0.23004637]],
                          [[ 0.50813986,  0.54309974],
                           [ 0.64082444,  0.67101435]]],
                         [[[-0.98053589, -1.03143541],
                           [-1.19128892, -1.24695841]],
                          [[ 0.69108355,  0.66880383],
                           [ 0.59480972,  0.56776003]],
                          [[ 2.36270298,  2.36904306],
                           [ 2.38090835,  2.38247847]]]])

# Сравните ваш результат с нашим; разница должна быть порядка e-8
print('Проверка conv_forward_naive')
print('разница: ', rel_error(out, correct_out))

## Отступление: обработка изображений с помощью свёрток

В качестве интересного способа проверить реализацию и лучше понять, какие операции могут выполнять сверточные слои, мы создадим входные данные, содержащие два изображения, и вручную зададим фильтры для распространённых операций обработки изображений (преобразование в оттенки серого и выделение границ). Прямой проход свёртки применит эти операции к каждому входному изображению. Затем мы сможем визуализировать результаты в качестве проверки.

In [ ]:
from imageio import imread
from PIL import Image

kitten = imread('cs231n/notebook_images/kitten.jpg')
puppy = imread('cs231n/notebook_images/puppy.jpg')
# kitten широкое, а puppy уже квадратное
d = kitten.shape[1] - kitten.shape[0]
kitten_cropped = kitten[:, d//2:-d//2, :]

img_size = 200   # Уменьшите это значение, если выполнение идёт слишком медленно
resized_puppy = np.array(Image.fromarray(puppy).resize((img_size, img_size)))
resized_kitten = np.array(Image.fromarray(kitten_cropped).resize((img_size, img_size)))
x = np.zeros((2, 3, img_size, img_size))
x[0, :, :, :] = resized_puppy.transpose((2, 0, 1))
x[1, :, :, :] = resized_kitten.transpose((2, 0, 1))

# Создаём веса свёртки с 2 фильтрами размером 3x3
w = np.zeros((2, 3, 3, 3))

# Первый фильтр преобразует изображение в оттенки серого.
# Задаём каналы красного, зелёного и синего фильтра.
w[0, 0, :, :] = [[0, 0, 0], [0, 0.3, 0], [0, 0, 0]]
w[0, 1, :, :] = [[0, 0, 0], [0, 0.6, 0], [0, 0, 0]]
w[0, 2, :, :] = [[0, 0, 0], [0, 0.1, 0], [0, 0, 0]]

# Второй фильтр выделяет горизонтальные границы в синем канале.
w[1, 2, :, :] = [[1, 2, 1], [0, 0, 0], [-1, -2, -1]]

# Вектор смещений. Для фильтра оттенков серого смещение не нужно
# — для фильтра выделения границ мы добавим 128
# к каждому выходу, чтобы ничего не было отрицательным.
b = np.array([0, 128])

# Вычисляем результат свёртки каждого входа из x с каждым фильтром из w,
# добавляя смещение b и сохраняя результаты в out.
out, _ = conv_forward_naive(x, w, b, {'stride': 1, 'pad': 1})

def imshow_no_ax(img, normalize=True):
    """ Tiny helper to show images as uint8 and remove axis labels """
    if normalize:
        img_max, img_min = np.max(img), np.min(img)
        img = 255.0 * (img - img_min) / (img_max - img_min)
    plt.imshow(img.astype('uint8'))
    plt.gca().axis('off')

# Показываем исходные изображения и результаты операции свёртки
plt.subplot(2, 3, 1)
imshow_no_ax(puppy, normalize=False)
plt.title('Исходное изображение')
plt.subplot(2, 3, 2)
imshow_no_ax(out[0, 0])
plt.title('Оттенки серого')
plt.subplot(2, 3, 3)
imshow_no_ax(out[0, 1])
plt.title('Границы')
plt.subplot(2, 3, 4)
imshow_no_ax(kitten_cropped, normalize=False)
plt.subplot(2, 3, 5)
imshow_no_ax(out[1, 0])
plt.subplot(2, 3, 6)
imshow_no_ax(out[1, 1])
plt.show()

# Свертка: обратный проход без оптимизаций
Реализуйте обратный проход для операции свёртки в функции `conv_backward_naive` в файле `cs231n/layers.py`. И снова вам не нужно слишком беспокоиться о вычислительной эффективности.

Когда закончите, запустите следующее, чтобы проверить обратный проход с помощью численной проверки градиента.

_Подсказка: https://deeplearning.cs.cmu.edu/F21/document/recitation/Recitation5/CNN_Backprop_Recitation_5_F21.pdf — это поможет получить общее представление. Реальная наивная реализация включает вложенные циклы `for` для непосредственного вычисления `dx_padded` с использованием индивидуальных производных из `dw` и `dout`._

In [ ]:
np.random.seed(231)
x = np.random.randn(4, 3, 5, 5)
w = np.random.randn(2, 3, 3, 3)
b = np.random.randn(2,)
dout = np.random.randn(4, 2, 5, 5)
conv_param = {'stride': 1, 'pad': 1}

dx_num = eval_numerical_gradient_array(lambda x: conv_forward_naive(x, w, b, conv_param)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: conv_forward_naive(x, w, b, conv_param)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: conv_forward_naive(x, w, b, conv_param)[0], b, dout)

out, cache = conv_forward_naive(x, w, b, conv_param)
dx, dw, db = conv_backward_naive(dout, cache)

# Ошибки должны быть порядка e-8 или меньше.
print('Проверка функции conv_backward_naive')
print('ошибка dx: ', rel_error(dx, dx_num))
print('ошибка dw: ', rel_error(dw, dw_num))
print('ошибка db: ', rel_error(db, db_num))

# Max-Pooling: прямой проход без оптимизаций
Реализуйте прямой проход для операции max-pooling в функции `max_pool_forward_naive` в файле `cs231n/layers.py`. И снова не беспокойтесь слишком сильно о вычислительной эффективности.

Проверьте реализацию, запустив следующее:

In [ ]:
x_shape = (2, 3, 4, 4)
x = np.linspace(-0.3, 0.4, num=np.prod(x_shape)).reshape(x_shape)
pool_param = {'pool_width': 2, 'pool_height': 2, 'stride': 2}

out, _ = max_pool_forward_naive(x, pool_param)

correct_out = np.array([[[[-0.26315789, -0.24842105],
                          [-0.20421053, -0.18947368]],
                         [[-0.14526316, -0.13052632],
                          [-0.08631579, -0.07157895]],
                         [[-0.02736842, -0.01263158],
                          [ 0.03157895,  0.04631579]]],
                        [[[ 0.09052632,  0.10526316],
                          [ 0.14947368,  0.16421053]],
                         [[ 0.20842105,  0.22315789],
                          [ 0.26736842,  0.28210526]],
                         [[ 0.32631579,  0.34105263],
                          [ 0.38526316,  0.4       ]]]])

# Сравните ваш результат с нашим. Разница должна быть порядка e-8.
print('Проверка функции max_pool_forward_naive:')
print('разница: ', rel_error(out, correct_out))

# Max-Pooling: обратный проход без оптимизаций
Реализуйте обратный проход для операции max-pooling в функции `max_pool_backward_naive` в файле `cs231n/layers.py`. Вам не нужно беспокоиться о вычислительной эффективности.

Проверьте реализацию с помощью численной проверки градиента, запустив следующее:

In [ ]:
np.random.seed(231)
x = np.random.randn(3, 2, 8, 8)
dout = np.random.randn(3, 2, 4, 4)
pool_param = {'pool_height': 2, 'pool_width': 2, 'stride': 2}

dx_num = eval_numerical_gradient_array(lambda x: max_pool_forward_naive(x, pool_param)[0], x, dout)

out, cache = max_pool_forward_naive(x, pool_param)
dx = max_pool_backward_naive(dout, cache)

# Ошибка должна быть порядка e-12
print('Проверка функции max_pool_backward_naive:')
print('ошибка dx: ', rel_error(dx, dx_num))

# Быстрые слои

Сделать слои свёртки и pooling быстрыми может быть непросто. Чтобы избавить вас от лишних трудностей, мы предоставили быстрые реализации прямого и обратного проходов для слоёв свёртки и pooling в файле `cs231n/fast_layers.py`.

### Выполните приведённую ниже ячейку, сохраните ноутбук и перезапустите среду выполнения
Быстрая реализация свёртки зависит от расширения Cython; чтобы скомпилировать его, выполните ячейку ниже. Затем сохраните ноутбук Colab (`File > Save`) и **перезапустите среду выполнения** (`Runtime > Restart runtime`). После этого можно повторно выполнить предыдущие ячейки сверху вниз и пропустить ячейку ниже, потому что она нужна только один раз для этапа компиляции.

In [ ]:
# Не забудьте перезапустить среду выполнения после выполнения этой ячейки!
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/
!python setup.py build_ext --inplace
%cd /content/drive/My\ Drive/$FOLDERNAME/

API для быстрых версий слоёв свёртки и pooling полностью совпадает с наивными версиями, которые вы реализовали выше: прямой проход принимает данные, веса и параметры и возвращает выходные значения и объект кэша; обратный проход принимает производные сверху и объект кэша и возвращает градиенты по данным и весам.

**Примечание:** Быстрая реализация pooling будет работать оптимально только если области pooling не перекрываются и покрывают вход целиком. Если эти условия не выполнены, быстрая реализация pooling будет не намного быстрее наивной реализации.

Сравнить производительность наивных и быстрых версий этих слоёв можно, запустив следующее:

In [ ]:
# Относительные ошибки должны быть порядка e-9 или меньше.
from cs231n.fast_layers import conv_forward_fast, conv_backward_fast
from time import time
np.random.seed(231)
x = np.random.randn(100, 3, 31, 31)
w = np.random.randn(25, 3, 3, 3)
b = np.random.randn(25,)
dout = np.random.randn(100, 25, 16, 16)
conv_param = {'stride': 2, 'pad': 1}

t0 = time()
out_naive, cache_naive = conv_forward_naive(x, w, b, conv_param)
t1 = time()
out_fast, cache_fast = conv_forward_fast(x, w, b, conv_param)
t2 = time()

print('Проверка conv_forward_fast:')
print('Наивно: %fs' % (t1 - t0))
print('Быстро: %fs' % (t2 - t1))
print('Ускорение: %fx' % ((t1 - t0) / (t2 - t1)))
print('Разница: ', rel_error(out_naive, out_fast))

t0 = time()
dx_naive, dw_naive, db_naive = conv_backward_naive(dout, cache_naive)
t1 = time()
dx_fast, dw_fast, db_fast = conv_backward_fast(dout, cache_fast)
t2 = time()

print('\nПроверка conv_backward_fast:')
print('Наивно: %fs' % (t1 - t0))
print('Быстро: %fs' % (t2 - t1))
print('Ускорение: %fx' % ((t1 - t0) / (t2 - t1)))
print('разница dx: ', rel_error(dx_naive, dx_fast))
print('разница dw: ', rel_error(dw_naive, dw_fast))
print('разница db: ', rel_error(db_naive, db_fast))

In [ ]:
# Относительные ошибки должны быть близки к 0.0.
from cs231n.fast_layers import max_pool_forward_fast, max_pool_backward_fast
np.random.seed(231)
x = np.random.randn(100, 3, 32, 32)
dout = np.random.randn(100, 3, 16, 16)
pool_param = {'pool_height': 2, 'pool_width': 2, 'stride': 2}

t0 = time()
out_naive, cache_naive = max_pool_forward_naive(x, pool_param)
t1 = time()
out_fast, cache_fast = max_pool_forward_fast(x, pool_param)
t2 = time()

print('Проверка pool_forward_fast:')
print('Наивно: %fs' % (t1 - t0))
print('быстро: %fs' % (t2 - t1))
print('ускорение: %fx' % ((t1 - t0) / (t2 - t1)))
print('разница: ', rel_error(out_naive, out_fast))

t0 = time()
dx_naive = max_pool_backward_naive(dout, cache_naive)
t1 = time()
dx_fast = max_pool_backward_fast(dout, cache_fast)
t2 = time()

print('\nПроверка pool_backward_fast:')
print('Наивно: %fs' % (t1 - t0))
print('быстро: %fs' % (t2 - t1))
print('ускорение: %fx' % ((t1 - t0) / (t2 - t1)))
print('разница dx: ', rel_error(dx_naive, dx_fast))

# Сверточные «сэндвич»-слои
В предыдущем задании мы познакомились с концепцией «сэндвич»-слоёв, которые объединяют несколько операций в часто используемые паттерны. В файле `cs231n/layer_utils.py` вы найдёте sandwich-слои, которые реализуют несколько распространённых паттернов для сверточных сетей. Запустите ячейки ниже, чтобы проверить их использование.

In [ ]:
from cs231n.layer_utils import conv_relu_pool_forward, conv_relu_pool_backward
np.random.seed(231)
x = np.random.randn(2, 3, 16, 16)
w = np.random.randn(3, 3, 3, 3)
b = np.random.randn(3,)
dout = np.random.randn(2, 3, 8, 8)
conv_param = {'stride': 1, 'pad': 1}
pool_param = {'pool_height': 2, 'pool_width': 2, 'stride': 2}

out, cache = conv_relu_pool_forward(x, w, b, conv_param, pool_param)
dx, dw, db = conv_relu_pool_backward(dout, cache)

dx_num = eval_numerical_gradient_array(lambda x: conv_relu_pool_forward(x, w, b, conv_param, pool_param)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: conv_relu_pool_forward(x, w, b, conv_param, pool_param)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: conv_relu_pool_forward(x, w, b, conv_param, pool_param)[0], b, dout)

# Относительные ошибки должны быть порядка e-8 или меньше
print('Проверка conv_relu_pool')
print('ошибка dx: ', rel_error(dx_num, dx))
print('ошибка dw: ', rel_error(dw_num, dw))
print('ошибка db: ', rel_error(db_num, db))

In [ ]:
from cs231n.layer_utils import conv_relu_forward, conv_relu_backward
np.random.seed(231)
x = np.random.randn(2, 3, 8, 8)
w = np.random.randn(3, 3, 3, 3)
b = np.random.randn(3,)
dout = np.random.randn(2, 3, 8, 8)
conv_param = {'stride': 1, 'pad': 1}

out, cache = conv_relu_forward(x, w, b, conv_param)
dx, dw, db = conv_relu_backward(dout, cache)

dx_num = eval_numerical_gradient_array(lambda x: conv_relu_forward(x, w, b, conv_param)[0], x, dout)
dw_num = eval_numerical_gradient_array(lambda w: conv_relu_forward(x, w, b, conv_param)[0], w, dout)
db_num = eval_numerical_gradient_array(lambda b: conv_relu_forward(x, w, b, conv_param)[0], b, dout)

# Относительные ошибки должны быть порядка e-8 или меньше
print('Проверка conv_relu:')
print('ошибка dx: ', rel_error(dx_num, dx))
print('ошибка dw: ', rel_error(dw_num, dw))
print('ошибка db: ', rel_error(db_num, db))

# Трёхслойная сверточная сеть
Теперь, когда вы реализовали все необходимые слои, мы можем собрать их вместе в простую сверточную сеть.

Откройте файл `cs231n/classifiers/cnn.py` и завершите реализацию класса `ThreeLayerConvNet`. Не забывайте, что в реализации можно использовать быстрые/sandwich-слои (они уже импортированы для вас). Запускайте следующие ячейки, чтобы помочь себе в отладке:

## Проверка потерь
После построения новой сети одной из первых вещей, которые нужно сделать, является проверка потерь. Когда мы используем функцию потерь softmax, ожидается, что потеря для случайных весов (и без регуляризации) будет примерно равна `log(C)` для `C` классов. Когда мы добавляем регуляризацию, потеря должна немного увеличиться.

In [ ]:
model = ThreeLayerConvNet()

N = 50
X = np.random.randn(N, 3, 32, 32)
y = np.random.randint(10, size=N)

loss, grads = model.loss(X, y)
print('Начальная потеря (без регуляризации): ', loss)

model.reg = 0.5
loss, grads = model.loss(X, y)
print('Начальная потеря (с регуляризацией): ', loss)

## Проверка градиента
Когда потеря выглядит разумно, используйте численную проверку градиента, чтобы убедиться, что обратный проход корректен. При использовании численной проверки градиента следует использовать небольшое количество искусственных данных и небольшое число нейронов на каждом слое. Примечание: корректные реализации могут всё ещё иметь относительные ошибки до порядка e-2.

In [ ]:
num_inputs = 2
input_dim = (3, 16, 16)
reg = 0.0
num_classes = 10
np.random.seed(231)
X = np.random.randn(num_inputs, *input_dim)
y = np.random.randint(num_classes, size=num_inputs)

model = ThreeLayerConvNet(
    num_filters=3,
    filter_size=3,
    input_dim=input_dim,
    hidden_dim=7,
    dtype=np.float64
)
loss, grads = model.loss(X, y)
# Ошибки должны быть небольшими, но корректные реализации могут иметь
# относительные ошибки до порядка e-2
for param_name in sorted(grads):
    f = lambda _: model.loss(X, y)[0]
    param_grad_num = eval_numerical_gradient(f, model.params[param_name], verbose=False, h=1e-6)
    e = rel_error(param_grad_num, grads[param_name])
    print('%s максимальная относительная ошибка: %e' % (param_name, rel_error(param_grad_num, grads[param_name])))

## Переобучение на небольших данных
Хороший приём — обучать модель всего на нескольких примерах. Вы сможете переобучить небольшие наборы данных, что приведёт к очень высокой точности на обучении и сравнительно низкой точности на валидации.

In [ ]:
np.random.seed(231)

num_train = 100
small_data = {
  'X_train': data['X_train'][:num_train],
  'y_train': data['y_train'][:num_train],
  'X_val': data['X_val'],
  'y_val': data['y_val'],
}

model = ThreeLayerConvNet(weight_scale=1e-2)

solver = Solver(
    model,
    small_data,
    num_epochs=15,
    batch_size=50,
    update_rule='adam',
    optim_config={'learning_rate': 1e-3,},
    verbose=True,
    print_every=1
)
solver.train()

In [ ]:
# Выводим итоговую точность на обучающей выборке.
print(
    "Точность на небольших данных (обучение):",
    solver.check_accuracy(small_data['X_train'], small_data['y_train'])
)

In [ ]:
# Выводим итоговую точность на валидационной выборке.
print(
    "Small data validation accuracy:",
    solver.check_accuracy(small_data['X_val'], small_data['y_val'])
)

Построение графиков потерь, точности на обучении и точности на валидации должно показать заметное переобучение:

In [ ]:
plt.subplot(2, 1, 1)
plt.plot(solver.loss_history, 'o')
plt.xlabel('iteration')
plt.ylabel('потеря')

plt.subplot(2, 1, 2)
plt.plot(solver.train_acc_history, '-o')
plt.plot(solver.val_acc_history, '-o')
plt.legend(['train', 'val'], loc='upper left')
plt.xlabel('epoch')
plt.ylabel('точность')
plt.show()

## Обучение сети
Обучая трёхслойную сверточную сеть в течение одного периода, вы должны достичь точности на обучающей выборке более 40%:

In [ ]:
model = ThreeLayerConvNet(weight_scale=0.001, hidden_dim=500, reg=0.001)

solver = Solver(
    model,
    data,
    num_epochs=1,
    batch_size=50,
    update_rule='adam',
    optim_config={'learning_rate': 1e-3,},
    verbose=True,
    print_every=20
)
solver.train()

In [ ]:
# Выводим итоговую точность на обучающей выборке.
print(
    "Точность на полном наборе данных (обучение):",
    solver.check_accuracy(data['X_train'], data['y_train'])
)

In [ ]:
# Выводим итоговую точность на валидационной выборке.
print(
    "Full data validation accuracy:",
    solver.check_accuracy(data['X_val'], data['y_val'])
)

## Визуализация фильтров
Вы можете визуализировать фильтры первого слоя сверточной сети из обученной модели, запустив следующее:

In [ ]:
from cs231n.vis_utils import visualize_grid

grid = visualize_grid(model.params['W1'].transpose(0, 2, 3, 1))
plt.imshow(grid.astype('uint8'))
plt.axis('off')
plt.gcf().set_size_inches(5, 5)
plt.show()